# `webai.news` — User Guide

Fetches and synthesises financial news using a fan-out Tavily pipeline, returning validated Pydantic models for breaking news, catalyst analysis, and daily market digests.

```
webai/news.py
│
├── Models (all exported)
│   ├── NewsItem          single news event with headline, sentiment, relevance_score
│   ├── SectorMover       sector movement note for digest summaries
│   └── NewsDigest        structured daily market digest
│
└── NewsResearcher(model, ...)
    ├── fetch_breaking_news(tickers, days_back=1)                    → list[NewsItem]
    ├── fetch_macro_news(days_back=1)                                → list[NewsItem]
    ├── fetch_catalyst_news(ticker, days_back=7)                     → list[NewsItem]
    └── fetch_daily_digest(tickers, sectors, date, days_back=1)      → NewsDigest
```

| Method | Returns | API keys needed |
|---|---|---|
| Model schema inspection | JSON schema | none |
| `fetch_breaking_news` | `list[NewsItem]` | TAVILY_API_KEY + OPENAI_API_KEY |
| `fetch_macro_news` | `list[NewsItem]` | TAVILY_API_KEY + OPENAI_API_KEY |
| `fetch_catalyst_news` | `list[NewsItem]` | TAVILY_API_KEY + OPENAI_API_KEY |
| `fetch_daily_digest` | `NewsDigest` | TAVILY_API_KEY + OPENAI_API_KEY |

**Prerequisites:**
- Sections 1–2 (setup, model schemas) run **without** API keys.
- Sections 3–8 require both `TAVILY_API_KEY` and `OPENAI_API_KEY` in `.env`.

## Sections
1. [Setup](#1-setup)
2. [Output model schemas](#2-output-model-schemas)
3. [Initialization](#3-initialization)
4. [fetch_breaking_news](#4-fetch_breaking_news)
5. [fetch_macro_news](#5-fetch_macro_news)
6. [fetch_catalyst_news](#6-fetch_catalyst_news)
7. [fetch_daily_digest](#7-fetch_daily_digest)
8. [End-to-end pattern](#8-end-to-end-pattern)
9. [Error handling reference](#9-error-handling-reference)

## 1 — Setup <a id="1-setup"></a>

In [ ]:
import json
import logging
import os

from dotenv import load_dotenv

from webai.news import NewsDigest, NewsItem, NewsResearcher, SectorMover

load_dotenv()

logging.basicConfig(
    level=logging.WARNING,  # flip to DEBUG to see the full pipeline trace
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    force=True,
)

_HAS_TAVILY = bool(os.environ.get("TAVILY_API_KEY"))
_HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
_HAS_KEYS   = _HAS_TAVILY and _HAS_OPENAI

print(f"TAVILY_API_KEY : {'set' if _HAS_TAVILY else 'MISSING'}")
print(f"OPENAI_API_KEY : {'set' if _HAS_OPENAI else 'MISSING'}")
print(f"Live sections  : {'enabled' if _HAS_KEYS else 'SKIPPED — set both keys in .env'}")

## 2 — Output model schemas <a id="2-output-model-schemas"></a>

All three exported models are Pydantic `BaseModel` subclasses. You can inspect their JSON schemas without any API keys.

### 2a — `NewsItem`

The core output unit — one per distinct news event.

| Field | Type | Notes |
|---|---|---|
| `headline` | `str` | Exact or paraphrased headline |
| `summary` | `str` | 2–4 sentence market-relevant summary |
| `source_url` | `str` | Article URL; `""` if unavailable |
| `published_date` | `str` | YYYY-MM-DD or `""` if not determinable |
| `tickers_mentioned` | `list[str]` | Uppercase symbols mentioned or clearly implied |
| `event_type` | `Literal[...]` | `earnings / guidance / analyst_action / macro / regulatory / m_and_a / insider / technical / other` |
| `sentiment` | `Literal[...]` | `bullish / bearish / neutral` |
| `relevance_score` | `float` (0–1) | 0.9+ reserved for clearly market-moving breaking news |

`relevance_score` is the primary sort key — all fetch methods return items sorted descending by this field.

In [ ]:
# Inspect the full JSON schema — no API key needed
print(json.dumps(NewsItem.model_json_schema(), indent=2))

### 2b — `SectorMover`

A sector showing notable movement, embedded in `NewsDigest.sector_movers`.

| Field | Type | Notes |
|---|---|---|
| `sector` | `str` | Sector name |
| `note` | `str` | 1–2 sentence note on why this sector is moving |

### 2c — `NewsDigest`

The top-level output of `fetch_daily_digest` — a structured view of one trading day.

| Field | Type | Notes |
|---|---|---|
| `date` | `str` | YYYY-MM-DD date this digest covers |
| `top_stories` | `list[NewsItem]` | 3–7 highest-relevance items |
| `market_theme` | `str` | 1–3 sentences on the dominant market narrative |
| `sector_movers` | `list[SectorMover]` | Up to 5 notable sectors |
| `macro_events` | `list[str]` | Fed decisions, CPI releases, geopolitical events |
| `watchlist_alerts` | `list[str]` | Short alert strings for watchlist tickers/themes; empty if no watchlist supplied |

In [ ]:
print(json.dumps(NewsDigest.model_json_schema(), indent=2))

## 3 — Initialization <a id="3-initialization"></a>

`NewsResearcher` wraps all four fetch methods. Build it once and reuse — the two structured-output chains (`_news_chain` and `_digest_chain`) are constructed at init time.

### Constructor parameters

| Parameter | Default | Notes |
|---|---|---|
| `model` | *(required)* | Any LangChain `BaseChatModel` with `with_structured_output` support |
| `tavily_api_key` | `$TAVILY_API_KEY` | Falls back to env var; **required** — raises `ValueError` if absent |
| `openai_api_key` | `$OPENAI_API_KEY` | Passed to the underlying `WebSearcher` |
| `max_results_per_query` | `5` | Tavily results per fan-out query |
| `trusted_domains` | default allowlist | Override the 22-entry `TRUSTED_DOMAINS` from `webai.ticker` |
| `filter_by_domain` | `True` | Falls back to unfiltered if all results removed; set `False` to skip entirely |
| `max_workers` | `8` | Thread pool size for parallel Tavily searches |
| `debug` | `False` | Sets this module's logger to DEBUG + attaches StreamHandler |

> **Invariant:** `__init__` raises `ValueError` immediately if `TAVILY_API_KEY` is missing. Unlike `WebSearcher`, `NewsResearcher` has no OpenAI fallback — Tavily is the sole search backend.

In [ ]:
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    from langchain_openai import ChatOpenAI

    model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    nr = NewsResearcher(
        model=model,
        max_results_per_query=3,  # keep calls fast in the guide
        debug=False,
    )

    print(f"filter_by_domain  : {nr.filter_by_domain}")
    print(f"max_workers       : {nr.max_workers}")
    print(f"trusted_domains   : {len(nr.trusted_domains)} entries")
    print(f"First few domains : {nr.trusted_domains[:5]}")

## 4 — `fetch_breaking_news` <a id="4-fetch_breaking_news"></a>

Fetches breaking news for one or more ticker symbols from the last N days.

**Pipeline:**
1. For each ticker, build a `"{TICKER} breaking news"` anchor query and fan it out via LLM query translation
2. All query variants across all tickers are pooled and deduplicated
3. All queries run in parallel against Tavily `topic="news"` with `days_back` time filter
4. Results are deduplicated → domain-filtered → passed to the news extraction chain
5. Returned items are sorted by `relevance_score` descending

**When to use vs alternatives:**
- `fetch_breaking_news` — most recent headlines, broad per-ticker coverage; best for `days_back=1`
- `fetch_catalyst_news` — deeper dive on one ticker's catalysts (earnings, upgrades); best for `days_back=3–14`
- `fetch_daily_digest` — combined macro + ticker + sector summary for a full trading day

In [ ]:
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    breaking = nr.fetch_breaking_news(["NVDA", "AAPL"], days_back=1)

    print(f"Items returned : {len(breaking)}")
    print()
    for item in breaking[:3]:
        print(f"[{item.relevance_score:.2f}] {item.headline}")
        print(f"       sentiment    : {item.sentiment}")
        print(f"       event_type   : {item.event_type}")
        print(f"       tickers      : {item.tickers_mentioned}")
        print(f"       source       : {item.source_url}")
        print()

In [ ]:
# Filter by sentiment or event type for downstream use
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    bearish = [i for i in breaking if i.sentiment == "bearish"]
    high_impact = [i for i in breaking if i.relevance_score >= 0.7]

    print(f"Bearish items  : {len(bearish)}")
    print(f"High-impact    : {len(high_impact)}  (relevance_score >= 0.7)")

    print("\nJSON of top item:")
    if breaking:
        print(breaking[0].model_dump_json(indent=2))

## 5 — `fetch_macro_news` <a id="5-fetch_macro_news"></a>

Fetches broad macro-economic news — Federal Reserve decisions, CPI releases, interest rate movements, inflation data — without targeting any specific ticker.

The anchor query is fixed: `"market macro news Federal Reserve interest rates inflation"`. It is fanned out via the same LLM query translation pipeline as all other methods.

**When to use:** Pair with `fetch_breaking_news` on the same `days_back` window to get a complete picture of both company-specific and macro-level news before building a digest manually.

In [ ]:
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    macro = nr.fetch_macro_news(days_back=1)

    print(f"Items returned : {len(macro)}")
    print()
    for item in macro[:3]:
        print(f"[{item.relevance_score:.2f}] {item.headline}")
        print(f"       event_type : {item.event_type}")
        print(f"       sentiment  : {item.sentiment}")
        print(f"       summary    : {item.summary[:120]}...")
        print()

## 6 — `fetch_catalyst_news` <a id="6-fetch_catalyst_news"></a>

Fetches catalyst-specific news for a **single** ticker: earnings results, analyst upgrades/downgrades, product launches, partnerships.

**Key differences from `fetch_breaking_news`:**

| Aspect | `fetch_breaking_news` | `fetch_catalyst_news` |
|---|---|---|
| Tickers | Multiple | Single |
| Tavily topic | `"news"` | `"finance"` (better for structured financial content) |
| Default `days_back` | 1 | 7 |
| Post-filter | None | Keeps items where ticker in `tickers_mentioned` OR `relevance_score >= 0.5` |

The post-filter prevents macro noise from dominating a ticker-specific query — items that mention the ticker explicitly or score ≥ 0.5 are kept; others are dropped.

In [ ]:
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    catalysts = nr.fetch_catalyst_news("NVDA", days_back=7)

    print(f"Items returned : {len(catalysts)}")
    print()
    for item in catalysts[:4]:
        print(f"[{item.relevance_score:.2f}] {item.headline}")
        print(f"       event_type      : {item.event_type}")
        print(f"       sentiment       : {item.sentiment}")
        print(f"       tickers_mentioned: {item.tickers_mentioned}")
        print()

In [ ]:
# Group by event_type to understand the catalyst mix
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    from collections import Counter

    type_counts = Counter(item.event_type for item in catalysts)
    print("Event type breakdown:")
    for event_type, count in type_counts.most_common():
        print(f"  {event_type:20s} : {count}")

## 7 — `fetch_daily_digest` <a id="7-fetch_daily_digest"></a>

Produces a structured `NewsDigest` for a single trading day by combining macro news, per-ticker queries, and per-sector queries in one parallel search pass.

**Query construction:**
- Always includes a macro anchor: `"market macro news Federal Reserve interest rates inflation"`
- Adds `"{TICKER} latest news"` for each ticker in the watchlist
- Adds `"{SECTOR} sector news outlook"` for each sector in the list

**Return vs fetch methods:** Unlike `fetch_breaking_news` and `fetch_macro_news`, this method calls a **digest chain** (`NewsDigest` structured output) rather than the news-extraction chain. The LLM is prompted as a senior market strategist synthesising the full day's coverage into one coherent view.

> **Invariant:** `fetch_daily_digest` raises `RuntimeError` on LLM synthesis failure. It never returns `None` or a partial object.

In [ ]:
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    digest = nr.fetch_daily_digest(
        tickers=["NVDA", "AAPL"],
        sectors=["Technology", "Energy"],
        days_back=1,
    )

    print(f"date          : {digest.date}")
    print(f"market_theme  : {digest.market_theme}")
    print(f"top_stories   : {len(digest.top_stories)}")
    print(f"sector_movers : {len(digest.sector_movers)}")
    print(f"macro_events  : {len(digest.macro_events)}")
    print(f"alerts        : {len(digest.watchlist_alerts)}")

In [ ]:
# Drill into the digest fields
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    print("Top stories:")
    for story in digest.top_stories:
        print(f"  [{story.relevance_score:.2f}] {story.headline}  ({story.sentiment})")

    print("\nSector movers:")
    for mover in digest.sector_movers:
        print(f"  {mover.sector:20s} : {mover.note[:80]}...")

    print("\nMacro events:")
    for event in digest.macro_events:
        print(f"  • {event}")

    if digest.watchlist_alerts:
        print("\nWatchlist alerts:")
        for alert in digest.watchlist_alerts:
            print(f"  ! {alert}")

In [ ]:
# Full JSON export — useful for logging, persistence, or downstream consumers
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    print(digest.model_dump_json(indent=2))

## 8 — End-to-end pattern <a id="8-end-to-end-pattern"></a>

A realistic morning-briefing workflow: fetch a daily digest, extract high-impact items, and route them to a per-ticker catalyst check for any ticker with a high-relevance alert.

This pattern combines all four public methods in a single `NewsResearcher` instance, so both chains are built only once.

In [ ]:
def morning_briefing(
    nr: NewsResearcher,
    watchlist: list[str],
    sectors: list[str],
    days_back: int = 1,
    catalyst_threshold: float = 0.75,
) -> dict:
    """
    Step 1 — daily digest for macro + watchlist overview.
    Step 2 — for each ticker flagged in top_stories with score >= threshold,
              run a catalyst deep-dive.
    Returns a dict with 'digest' and 'catalysts' keys.
    """
    # Step 1: full-day digest
    day_digest = nr.fetch_daily_digest(
        tickers=watchlist,
        sectors=sectors,
        days_back=days_back,
    )

    # Step 2: identify alerted tickers from high-relevance top stories
    alerted: set[str] = set()
    for story in day_digest.top_stories:
        if story.relevance_score >= catalyst_threshold:
            alerted.update(story.tickers_mentioned)
    alerted = {t for t in alerted if t in [w.upper() for w in watchlist]}

    # Step 3: per-ticker catalyst deep-dive
    catalyst_map: dict[str, list] = {}
    for ticker in alerted:
        catalyst_map[ticker] = nr.fetch_catalyst_news(ticker, days_back=days_back + 6)

    return {"digest": day_digest, "catalysts": catalyst_map}


# --- run the briefing ---
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    briefing = morning_briefing(
        nr,
        watchlist=["NVDA", "AAPL", "MSFT"],
        sectors=["Technology", "Semiconductors"],
        days_back=1,
        catalyst_threshold=0.75,
    )

    d = briefing["digest"]
    print(f"Date          : {d.date}")
    print(f"Market theme  : {d.market_theme}")
    print(f"Top stories   : {len(d.top_stories)}")
    print(f"Catalyst tickers triggered: {list(briefing['catalysts'].keys())}")
    print()

    for ticker, items in briefing["catalysts"].items():
        print(f"--- {ticker} catalyst deep-dive ({len(items)} items) ---")
        for item in items[:2]:
            print(f"  [{item.relevance_score:.2f}] {item.headline}")

## 9 — Error handling reference <a id="9-error-handling-reference"></a>

| Method | Guard condition | Exception |
|---|---|---|
| `NewsResearcher.__init__` | `TAVILY_API_KEY` missing | `ValueError` |
| `fetch_daily_digest` | LLM synthesis fails | `RuntimeError` |
| `fetch_daily_digest` | chain returns unexpected type | `RuntimeError` |
| `fetch_breaking_news` | `tickers=[]` | returns `[]` (not an error) |
| `_extract_news_items` | extraction chain fails | returns `[]`, logs WARNING (not raised) |

> **Fallback behaviour:** Domain filtering never raises — it returns an unfiltered set and logs WARNING when every result is excluded. Individual Tavily search failures are also swallowed (WARNING log) so a single bad query never aborts the pipeline.

In [ ]:
# ValueError: missing Tavily key at construction time (always runs — no live API needed)
import os as _os

_saved_tavily = _os.environ.pop("TAVILY_API_KEY", None)
try:
    from langchain_openai import ChatOpenAI as _ChatOpenAI
    NewsResearcher(model=_ChatOpenAI(model="gpt-4o-mini"))
except ValueError as e:
    print(f"NewsResearcher(no key) -> ValueError: {e}")
finally:
    if _saved_tavily:
        _os.environ["TAVILY_API_KEY"] = _saved_tavily

In [ ]:
# Empty tickers list returns [] immediately — no API call made
if not _HAS_KEYS:
    print("Skipping — requires TAVILY_API_KEY and OPENAI_API_KEY.")
else:
    result = nr.fetch_breaking_news([])
    print(f"fetch_breaking_news([]) -> {result!r}  (empty list, no error)")